There is a folder with name
```
C:\Users\kou12\University of Cambridge\Jose Lucas Taiyo Lee Rocha Santos - Cheeseboard\Raw Data\
```
Which have many folders with the name in the `YYYY-MM-DD` format
within each folder there is a subfolder called `movie`
And within that `movie` folder there are multiple files in the *.avi format.
Those *.avi file names are all in the format: `YYYY-MM-DD_{Animal_name}_trial_{n}.avi`

Below is a python code, that goes through all the sub-folder in the `Raw Data` folder,
And make a python dataframe, with columns:
* id (Animal name)
* date (in the YYYYMMDD format)

The script will also pick out two types of problems:
1. Unexpected `*.avi` filename: will raise any avi which voilate the filename convention `YYYY-MM-DD_{Animal_name}_trial_{n}.avi`
2. Non-sequential trial ID: Animal-dates with potentially missing trials, for some, maybe mistakes in the animal names:
Note: AF-20250608: Missing trial(s) in sequence: [1, 3, 4] means in AF-20250608 only trial 1,3,4 are present but 2 is missing. If there are repeating trials, like [1, 1, ... 7, 7, 8, 8] it is likely that the data is saved twice at different locations in the Raw Data folder

### Utils

In [1]:
import os
import pandas as pd
import re

def check_movie_folder(base_path, date_dir):
    '''
    This function checks the *.avi files in each date directory. It does 2 things:
    1. Check if all files are in the format `YYYY-MM-DD_{Animal}_trial_{n}.avi`
    2. Returns a dataframe with columns: animal_id, date, trial_id, for further processing
    '''

    movie_path = os.path.join(base_path, date_dir, 'movie')
    # Get all file names with extension .avi in the movie_path
    file_names = [f for f in os.listdir(movie_path) if f.endswith('.avi')]

    # Check if all of them have the format: YYYY-MM-DD_{Animal}_trial_{n}.avi
    pattern = re.compile(r"(\d{4}-\d{2}-\d{2})_([^_]+)_trial_(\d+)\.avi")
    for file_name in file_names:
        match = pattern.match(file_name)
        if not match:
            # If the file name start with a dot, skip it (hidden files)
            if file_name.startswith('.'):
                continue
            print(f"File name '{file_name}' in {movie_path} does not match the expected format.")

    # Get a dataframe with columns: animal_id, date, trial_id
    data = {
        'animal_id': [],
        'date': [],
        'trial_id': []
    }
    for file_name in file_names:
        match = pattern.match(file_name)
        if match:
            date, animal_id, trial_id = match.groups()
            data['animal_id'].append(animal_id)
            data['date'].append(date)
            data['trial_id'].append(int(trial_id))
    df = pd.DataFrame(data)
    return df

def check_tracking_folder(base_path, date_dir):
    '''
    This function checks the *.csv files in each date directory under movie/tracking. It does 2 things:
    1. Check if all files are in the format `YYYY-MM-DD_{Animal}_trial_{n}_positions.csv`
    2. Returns a dataframe with columns: animal_id, date, trial_id, for further processing
    '''
    tracking_path = os.path.join(base_path, date_dir, 'movie', 'tracking')
    # Get all file names with extention .csv in the tracking path
    file_names = [f for f in os.listdir(tracking_path) if f.endswith('.csv')]

    # Check if all of them have the format: YYYY-MM-DD_{Animal}_trial_{n}_positions.csv
    pattern = re.compile(r"(\d{4}-\d{2}-\d{2})_([^_]+)_trial_(\d+)_positions\.csv")
    for file_name in file_names:
        match = pattern.match(file_name)
        if not match:
            # If the file name start with a dot, skip it (hidden files)
            if file_name.startswith('.'):
                continue
            print(f"File name '{file_name}' in {tracking_path} does not match the expected format.")

    # Get a dataframe with columns: animal_id, date, trial_id
    data = {
        'animal_id': [],
        'date': [],
        'trial_id': []
    }
    for file_name in file_names:
        match = pattern.match(file_name)
        if match:
            date, animal_id, trial_id = match.groups()
            data['animal_id'].append(animal_id)
            data['date'].append(date)
            data['trial_id'].append(int(trial_id))
    df = pd.DataFrame(data)

    return df

def compare_dataframes(df_movie, df_tracking):
    '''
    This function compares two dataframes with columns: animal_id, date, trial_id
    and prints out the differences between them.
    '''
    # Find rows in df_movie that are not in df_tracking
    df_diff = pd.merge(df_movie, df_tracking, on=['animal_id', 'date', 'trial_id'], how='outer', indicator=True)
    df_only_in_movie = df_diff[df_diff['_merge'] == 'left_only']
    if len(df_only_in_movie) > 0:
        print("The following trials are in movie but not in tracking:")
        print(df_only_in_movie[['animal_id', 'date', 'trial_id']])
    # Find rows in df_tracking that are not in df_movie
    df_only_in_tracking = df_diff[df_diff['_merge'] == 'right_only']
    if len(df_only_in_tracking) > 0:
        print("The following trials are in tracking but not in movie:")
        print(df_only_in_tracking[['animal_id', 'date', 'trial_id']])

def check_consecutive_trials(df_ses, data_type = 'movie'):
    '''
    This function checks if all trial IDs for each animal are consecutive from 1 to max trial ID.
    It prints out any missing trial IDs for each animal.
    '''
    # Get unique animal_ids
    animal_ids = df_ses['animal_id'].unique()
    for animal_id in animal_ids:
        df_ani = df_ses[df_ses['animal_id'] == animal_id]
        # Get the max trial_id
        max_trial_id = df_ani['trial_id'].max()
        # Check if 1 to max_trial_id all exist
        missing_trials = []
        for i in range(1, max_trial_id + 1):
            if i not in df_ani['trial_id'].values:
                missing_trials.append(i)
        if len(missing_trials) > 0:
            print(f"Missing {data_type} files for {df_ani['date'].iloc[0]} {animal_id}: trial {missing_trials}")

def combine_dataframes(df_1, df_2):
    '''
    This function combines two dataframes with columns: animal_id, date, trial_id
    It drops exact duplicate rows and orders the rows by animal_id, date, trial_id
    '''
    df_combined = pd.concat([df_1, df_2], ignore_index=True)

    # Drop exact duplicate rows
    df_combined = df_combined.drop_duplicates()

    # Order rows by animal_id
    df_combined = df_combined.sort_values(by=['animal_id', 'date', 'trial_id']).reset_index(drop=True)
    # Make sure the trial_id is integer
    df_combined['trial_id'] = df_combined['trial_id'].astype(int)
    return df_combined

### Quality Check Routine

In [4]:
# Path to the base folder
base_path = r"E:\Cheeseboard-rawdata-editted_Stella_edit"

# List the subdirectories in the base folder
subdirs = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]

### QC1: `locations.csv` exists in all date folders

In [5]:
# Check if `locations.csv` exists in each subdirectory
i = 0
for subdir in subdirs:
    csv_path = os.path.join(base_path, subdir, 'locations.csv')
    if not os.path.isfile(csv_path):
        print(f"'locations.csv' not found in {subdir}")
        i += 1
if i == 0:
    print("All subdirectories contain 'locations.csv'")

All subdirectories contain 'locations.csv'


### QC2: All movie files have the corresponding tracking files, and vice versa

In [6]:
for subdir in subdirs:
    df_movie = check_movie_folder(base_path, subdir)
    df_tracking = check_tracking_folder(base_path, subdir)
    compare_dataframes(df_movie, df_tracking)

The following trials are in tracking but not in movie:
  animal_id        date  trial_id
0        AI  2025-07-24         1
2        AJ  2025-07-24         1
The following trials are in movie but not in tracking:
   animal_id        date  trial_id
10         I  2024-12-01         3
The following trials are in tracking but not in movie:
  animal_id        date  trial_id
0        AI  2025-07-24         1
1        AJ  2025-07-24         1
The following trials are in movie but not in tracking:
   animal_id        date  trial_id
16        AF  2025-06-24         1
The following trials are in movie but not in tracking:
   animal_id        date  trial_id
11        AJ  2025-07-11         4
The following trials are in tracking but not in movie:
  animal_id        date  trial_id
8        AD  2025-06-28        11


### QC3: All consequtive trial IDs exist

In [7]:
for subdir in subdirs:
    df_movie = check_movie_folder(base_path, subdir)
    df_tracking = check_tracking_folder(base_path, subdir)
    check_consecutive_trials(df_movie, data_type='movie')
    check_consecutive_trials(df_tracking, data_type='tracking')

Missing tracking files for 2024-12-01 I: trial [3]
Missing tracking files for 2025-06-24 AF: trial [1]
Missing movie files for 2025-10-11 AN: trial [9, 10]
Missing movie files for 2025-10-11 AM: trial [9, 10]
Missing movie files for 2025-10-11 AO: trial [9, 10]
Missing tracking files for 2025-10-11 AM: trial [9, 10]
Missing tracking files for 2025-10-11 AN: trial [9, 10]
Missing tracking files for 2025-10-11 AO: trial [9, 10]
Missing movie files for 2025-08-22 AN: trial [9, 10]
Missing movie files for 2025-08-22 AO: trial [9, 10]
Missing movie files for 2025-08-22 AM: trial [9, 10]
Missing tracking files for 2025-08-22 AO: trial [9, 10]
Missing tracking files for 2025-08-22 AM: trial [9, 10]
Missing tracking files for 2025-08-22 AN: trial [9, 10]
Missing tracking files for 2025-07-11 AJ: trial [4]
Missing movie files for 2025-07-16 AJ: trial [9, 10]
Missing movie files for 2025-07-16 AI: trial [9, 10]
Missing movie files for 2025-07-16 AL: trial [9, 10]
Missing tracking files for 2025-

### QC4: Make the trial/ session/ animal metadata

In [21]:
df_combined_all = pd.DataFrame(columns=['animal_id', 'date', 'trial_id'])
for subdir in subdirs:
    df_movie = check_movie_folder(base_path, subdir)
    df_tracking = check_tracking_folder(base_path, subdir)
    df_combined = combine_dataframes(df_movie, df_tracking)
    df_combined_all = pd.concat([df_combined_all, df_combined], ignore_index=True)

# Order by animal_id, date, trial_id
df_combined_all = df_combined_all.sort_values(by=['animal_id', 'date', 'trial_id']).reset_index(drop=True)
df_combined_all['trial_id'] = df_combined_all['trial_id'].astype(int)
df_combined_all


,animal_id,date,trial_id
0,A,2024-12-05,1
1,A,2024-12-05,2
2,A,2024-12-05,3
3,A,2024-12-05,4
4,A,2024-12-05,5
...,...,...,...
4105,T,2024-11-12,7
4106,T,2024-11-12,8
4107,T,2024-11-13,1
4108,T,2024-11-14,1


In [23]:
subdir = '2025-07-25'
df_movie = check_movie_folder(base_path, subdir)
df_tracking = check_tracking_folder(base_path, subdir)
df_combined = combine_dataframes(df_movie, df_tracking)
df_combined

,animal_id,date,trial_id
0,AI,2025-07-24,1
1,AI,2025-07-25,1
2,AJ,2025-07-24,1
3,AJ,2025-07-25,1
4,AL,2025-07-25,1
5,AM,2025-07-25,1
6,AM,2025-07-25,2
7,AM,2025-07-25,3
8,AM,2025-07-25,4
9,AM,2025-07-25,5


In [18]:
import numpy as np
np.unique(subdirs, return_counts=True)[1]

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [ ]:
df_combined_all

# Save the combined dataframe to a csv file
df_combined_all.to_csv('trial_data.csv', index=False)

In [35]:
animal_ids = df_combined_all['animal_id'].unique()

In [36]:
# Make a dataframe with each row representing a unique (animal_id)

df_summary = pd.DataFrame(columns=['animal_id', 'genotype'])

df_summary['animal_id'] = animal_ids

# Save the summary dataframe to a csv file
df_summary.to_csv('genotypes.csv', index=False)

## Session metadata

In [41]:
df_combined_all
# Delete the trial_id column for summary purposes
df_combined_session = df_combined_all.drop(columns=['trial_id'])

In [42]:
df_combined_session = df_combined_session.drop_duplicates().reset_index(drop=True)

In [44]:
df_combined_session
# Save as session.csv
df_combined_session.to_csv('session_data.csv', index=False)

In [1]:
import os
import pandas as pd
import re

# Path to your Raw Data folder

base_path = r"E:\Cheeseboard-rawdata-editted_Stella_edit"

# Regex to capture: YYYY-MM-DD_{Animal}_trial_{n}.avi
pattern = re.compile(r"(\d{4}-\d{2}-\d{2})_([^_]+)_trial_(\d+)\.avi")

records = []

# Walk through Raw Data folder
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".avi"):
            m = pattern.match(file)
            if m:
                date_str, animal_name, trial_num = m.groups()
                date = date_str.replace("-", "")  # YYYYMMDD
                # if date == '20250607' and animal_name == 'AD':
                #     print(f"Found: {file}")
                trial_num = int(trial_num)
                records.append({
                    "id": animal_name,
                    "date": date,
                    "trial": trial_num
                })
            else:
                print(f"⚠️ Filename format unexpected: {file}")

# Make DataFrame
df_trials = pd.DataFrame(records)

# Group by id/date to compute number of trials
def check_trials(trials, animal_name, date):
    trials_sorted = sorted(trials)
    n_trials = len(trials_sorted)
    max_trial = max(trials_sorted)
    if trials_sorted != list(range(1, max_trial + 1)):
        print(f"⚠️ {animal_name}-{date}: Missing trial(s) in sequence: {trials_sorted}")
    return n_trials

trial_info = (
    df_trials.groupby(["id", "date"])["trial"]
    .apply(list)
    .reset_index()
)

trial_info["n_trials"] = trial_info.apply(
    lambda row: check_trials(row["trial"], row["id"], row["date"]), axis=1
)

# Merge into final df (one row per id/date)
df = trial_info.drop(columns="trial").sort_values(by=["id", "date"]).reset_index(drop=True)

print(df.head())

# # Save CSV
# output_csv = os.path.join(base_path, "animal_data_with_trials.csv")
# df.to_csv(output_csv, index=False)
# print(f"✅ CSV saved to {output_csv}")


⚠️ Filename format unexpected: ._2025-10-03_AO_trial_1.avi
⚠️ Filename format unexpected: ._2025-10-03_AO_trial_2.avi
⚠️ Filename format unexpected: ._2025-10-04_AO_trial_2.avi
⚠️ Filename format unexpected: ._2025-10-04_AO_trial_1.avi
⚠️ Filename format unexpected: ._2025-10-05_AN_trial_1.avi
⚠️ Filename format unexpected: ._2025-10-05_AM_trial_1.avi
⚠️ Filename format unexpected: ._2025-10-05_AO_trial_1.avi
⚠️ Filename format unexpected: ._2025-10-02_AO_trial_2.avi
⚠️ Filename format unexpected: ._2025-10-02_AO_trial_1.avi
⚠️ Filename format unexpected: ._2025-06-15_AD_trial_1.avi
⚠️ Filename format unexpected: ._2025-09-30_AN_trial_8.avi
⚠️ Filename format unexpected: ._2025-09-30_AO_trial_2.avi
⚠️ Filename format unexpected: ._2025-09-30_AO_trial_3.avi
⚠️ Filename format unexpected: ._2025-09-30_AO_trial_1.avi
⚠️ Filename format unexpected: ._2025-09-30_AM_trial_8.avi
⚠️ Filename format unexpected: ._2025-09-30_AO_trial_4.avi
⚠️ Filename format unexpected: ._2025-09-30_AO_trial_5.a

Now, with the df created, look into another folder, named:
```
C:\Users\kou12\University of Cambridge\Jose Lucas Taiyo Lee Rocha Santos - Cheeseboard\ChR2
```
One level down in this folder, there are 6 sub folders, the names of these subfolders does not matter.
But now one more level into each of the 6 subfolders, there are more sub folders, named in the format:
`YYYY-MM-DD{type}{number}`, 
`type` means the session type, `number` is the days since the start of that session type
into these `YYYY-MM-DD{type}{number}` subfolders there are many *.avi files, again they are named
YYYY-MM-DD_{Animal_name}_trial_{n}.avi

Below is a script that go through each `*.avi` files, and
Find the matching animals - date in the `df`.
Add a column `type` in `df` to save the session type information,
And also add a column `day_in_type` to save the `number` information

If that animals - date pair does not exist in `df`, print :
f'{id} - {date} does not exist'

In [7]:
import re

base_chr2 = r"C:\Users\kou12\University of Cambridge\Jose Lucas Taiyo Lee Rocha Santos - Cheeseboard\ChR2"

# --- Now scan ChR2 folder ---
# Regex for session folder: YYYY-MM-DD{type}{number}
session_pattern = re.compile(r"(\d{4}-\d{2}-\d{2})([A-Za-z]+)(\d+)")

new_records = []

# Go two levels down in ChR2
for subfolder in os.listdir(base_chr2):
    subfolder_path = os.path.join(base_chr2, subfolder)
    if os.path.isdir(subfolder_path):
        for session_folder in os.listdir(subfolder_path):
            session_path = os.path.join(subfolder_path, session_folder)
            if os.path.isdir(session_path):
                m = session_pattern.match(session_folder)
                if not m:
                    continue
                date_str, session_type, session_num = m.groups()
                date = date_str.replace("-", "")
                day_in_type = int(session_num)

                # Look for AVI files inside
                for file in os.listdir(session_path):
                    if file.endswith(".avi"):
                        parts = file.split("_")
                        if len(parts) >= 3:
                            animal_name = parts[1]

                            # Check if (id, date) exists in df
                            mask = (df["id"] == animal_name) & (df["date"] == date)
                            if mask.any():
                                # If the type and day_in_type columns values is not nan, check if they match
                                if "type" in df.columns and "day_in_type" in df.columns:
                                    existing_type = df.loc[mask, "type"].values[0]
                                    existing_day = df.loc[mask, "day_in_type"].values[0]
                                    if pd.notna(existing_type) and pd.notna(existing_day):
                                        if existing_type != session_type or existing_day != day_in_type:
                                            print(f"Conflict for {animal_name} on {date}: existing ({existing_type}, {existing_day}) vs new ({session_type}, {day_in_type})")
                                            continue  # Skip this entry
                                else:
                                    df["type"] = pd.NA
                                    df["day_in_type"] = pd.NA
                                # Update df with type and day_in_type
                                df.loc[mask, "type"] = session_type
                                df.loc[mask, "day_in_type"] = day_in_type
                            else:
                                print(f"{animal_name} - {date} does not exist")

In [10]:
df

,id,date,n_trials
0,9,20240518,1
1,A,20240305,1
2,A,20240306,3
3,A,20240307,3
4,A,20240308,1
...,...,...,...
769,T,20241111,8
770,T,20241112,8
771,T,20241113,1
772,T,20241114,1


In [13]:
# Save df as csv
df.to_csv("animal_sessions_RawDataEdited.csv", index=False)

Below code to check if the positions.csv and track.png exist for each avi videos

In [ ]:
import os
import re

# Base path
base_path = r"E:\Cheeseboard-rawdata-editted_Stella_edit"

# Regex to capture: YYYY-MM-DD_{Animal}_trial_{n}.avi
avi_pattern = re.compile(r"(\d{4}-\d{2}-\d{2})_([^_]+)_trial_(\d+)\.avi")

# Walk through Raw Data folder
for root, dirs, files in os.walk(base_path):
    # Only check "movie" subfolders
    if os.path.basename(root) != "movie":
        continue

    # Path to tracking folder
    tracking_path = os.path.join(root, "tracking")
    if not os.path.exists(tracking_path):
        print(f"⚠️ Tracking folder missing in: {root}")
        continue

    # Collect all tracking files
    tracking_files = set(os.listdir(tracking_path))

    for file in files:
        if file.endswith(".avi"):
            m = avi_pattern.match(file)
            if not m:
                print(f"⚠️ Unexpected AVI filename: {file}")
                continue

            date_str, animal_name, trial_num = m.groups()
            date = date_str.replace("-", "")
            base_name = f"{date_str}_{animal_name}_trial_{trial_num}"

            # Expected files inside tracking/
            csv_file = f"{base_name}_positions.csv"
            png_file = f"{base_name}_trace.png"

            if csv_file not in tracking_files:
                print(f"{animal_name}-{date}: missing csv file ({csv_file})")
            if png_file not in tracking_files:
                print(f"{animal_name}-{date}: missing png file ({png_file})")


I-20241201: missing csv file (2024-12-01_I_trial_3_positions.csv)
I-20241201: missing png file (2024-12-01_I_trial_3_trace.png)
AF-20250624: missing csv file (2025-06-24_AF_trial_1_positions.csv)
AF-20250624: missing png file (2025-06-24_AF_trial_1_trace.png)
AJ-20250711: missing csv file (2025-07-11_AJ_trial_4_positions.csv)
AJ-20250711: missing png file (2025-07-11_AJ_trial_4_trace.png)
AN-20250725: missing png file (2025-07-25_AN_trial_6_trace.png)


In [3]:
file_set

{'2025-08-23_AM_trial_1.avi',
 '2025-08-23_AM_trial_2.avi',
 '2025-08-23_AM_trial_3.avi',
 '2025-08-23_AN_trial_1.avi',
 '2025-08-23_AO_trial_1.avi',
 '2025-08-23_AO_trial_2.avi',
 'convert.py'}